# Gap：用 qust 检测价格跳空

来源参考：[Investopedia](https://www.investopedia.com/terms/g/gap.asp)


这篇 notebook 按 Investopedia 原页面的信息结构做完整中文改写，并把指标定义落成 `col(...).investopedia.xxx(...)` 的一行调用。能用 qust 现有 rolling、shift、select、with_cols、over 组合的就直接组合；需要 pivot/形态扫描的部分由 Rust helper 完成，Python 端不写 UDF。


## 1. Investopedia 原文内容完整改写：Gap

### 什么是 Gap
Gap 是价格图上相邻两个交易区间之间出现的空白。比如当前最低价仍高于上一根最高价，就是向上跳空；当前最高价低于上一根最低价，就是向下跳空。这说明两个时间段之间价格发生了不连续变化，中间价位没有成交覆盖。

### 为什么会出现缺口
缺口常由盘后消息、重大公告、宏观事件、供需突然变化或开盘集合竞价导致。股票日线缺口很常见；期货和连续交易市场也可能因为夜盘、节假日或流动性变化出现类似现象。

### 常见类型
常见分类包括 common gap、breakaway gap、runaway/measuring gap 和 exhaustion gap。普通缺口可能很快回补；突破缺口可能发生在重要区间被突破时；持续缺口可能出现在趋势中段；衰竭缺口可能出现在趋势末端。这些分类依赖位置、趋势和后续价格行为。

### 如何解读
向上缺口说明买方愿意在高于上一交易区间的位置成交；向下缺口说明卖方愿意在更低区域成交。但缺口不是天然看多或看空，关键要看它发生在哪里、是否突破重要区域、成交量如何、后续是否回补。

### 回补问题
很多交易者会关注 gap fill，也就是价格后来回到缺口区域。但并不是所有缺口都会回补，也不是回补时间固定。把“缺口一定回补”当成规则会很危险。

### 局限性
缺口检测本身很简单，但解释很复杂。不同市场交易时段不同，是否把隔夜、节假日、主力换月拼接当作缺口，都要先定义清楚。程序化时应明确使用严格区间缺口还是只比较 open 与前 close。

## 2. 从文章到 qust 算子的落地

qust 使用严格区间定义：`gap_up = current low > previous high`，`gap_down = current high < previous low`。同时输出带方向的 `gap_size` 和标准化 `gap_pct`。

## 3. qust 一行调用

```python
col("open", "high", "low", "close").investopedia.gap()
```

输入列顺序：`open, high, low, close`。

输出列：`gap_up`, `gap_down`, `gap_size`, `gap_pct`。

这些输出都保持和输入相同的行数，后面可以继续 `.with_cols(...)`、`.filter(...)`、`.monitor...`，也可以接 `.over("ticker", "ct")` 按合约独立计算。

In [1]:
import os
import sys

LOCAL_QUST_SOURCE = "/root/otters/otters-py/python"
if os.path.isdir(LOCAL_QUST_SOURCE) and LOCAL_QUST_SOURCE not in sys.path:
    sys.path.insert(0, LOCAL_QUST_SOURCE)

import qust as qs
import qust.future.future  # 注册 bt/stra/kline/fp 等金融命名空间
import qust.investopedia  # 注册 investopedia 命名空间
from qust import col, mark_shape
from qust._polars import pl

pl.Config.set_tbl_rows(16)
pl.Config.set_tbl_cols(28)

DATA_PATH = "/root/qust-py/examples/data/data_kline3.parquet"
PLOT_TICKER = "AP"


In [2]:
raw = pl.read_parquet(DATA_PATH).sort(["ticker", "ct", "datetime"])

base_contract = (
    raw
    .filter(pl.col("ticker") == PLOT_TICKER)
    .select("ct")
    .unique()
    .sort("ct")
    .get_column("ct")[0]
)

print("raw shape:", raw.shape)
print("tickers:", raw.select(pl.col("ticker").unique().sort()).to_series().to_list())
print("contract count:", raw.select("ticker", "ct").unique().height)
print("default plot ticker/ct:", PLOT_TICKER, base_contract)
raw.head(5)


raw shape: (408782, 8)
tickers: ['AP', 'RM', 'SA', 'al', 'eb', 'eg', 'fu', 'rb']
contract count: 141
default plot ticker/ct: AP 205


ticker,ct,datetime,open,high,low,close,volume
str,i32,datetime[ms],f64,f64,f64,f64,f64
"""AP""",205,2022-01-04 09:00:00,8394.0,8394.0,8392.0,8392.0,1100.0
"""AP""",205,2022-01-04 09:05:00,8385.0,8389.0,8348.0,8378.0,11169.0
"""AP""",205,2022-01-04 09:10:00,8375.0,8376.0,8298.0,8302.0,14001.0
"""AP""",205,2022-01-04 09:15:00,8301.0,8315.0,8271.0,8280.0,12839.0
"""AP""",205,2022-01-04 09:20:00,8279.0,8285.0,8243.0,8246.0,11496.0


## 4. 计算指标

下面用真实本地 K 线数据计算。对合约相关指标，示例都使用 `.over("ticker", "ct")`，表示每个品种、每个合约独立维护上下文，避免不同合约的数据串在一起。

In [3]:
indicator_expr = col("open", "high", "low", "close").investopedia.gap()
gap_data = col.with_cols(indicator_expr).over("ticker", "ct").calc_data(raw)
gap_data = gap_data.with_columns((pl.col("gap_up") | pl.col("gap_down")).alias("gap_signal"))
plot_data = (
    gap_data
    .filter((pl.col("ticker") == PLOT_TICKER) & (pl.col("ct") == base_contract))
    .sort("datetime")
    .head(1200)
)

summary = col(
    col("gap_up").cast(pl.UInt32).sum().alias("gap_up_count"),
    col("gap_down").cast(pl.UInt32).sum().alias("gap_down_count"),
    col("gap_pct").abs().mean().alias("avg_abs_gap_pct"),
).calc_data(gap_data)

print("plot shape:", plot_data.shape)
summary

plot shape: (1200, 13)


gap_up_count,gap_down_count,avg_abs_gap_pct
u32,u32,f64
2381,2038,0.000028


## 5. 用 monitor 画出来

图不是静态 PNG，而是 qust monitor 输出。你可以在 notebook 里放大、拖动、查看指标与 K 线的对应关系。

In [4]:
gap_plot = col(
    col("datetime", "open", "high", "low", "close", "volume")
        .monitor("gap_price", show_axis_label=True)
        .kline(),
    col("datetime", "low", "gap_up")
        .monitor("gap_price", show_axis_label=True)
        .mark(shape=mark_shape.triangle_up, color="#50fa7b", width=0.45),
    col("datetime", "high", "gap_down")
        .monitor("gap_price", show_axis_label=True)
        .mark(shape=mark_shape.triangle_down, color="#ff6b6b", width=0.45),
).monitor.make_monitor("black").monitor.add_grid([
    ["gap_price"],
]).runtime()

gap_plot.plot(plot_data, open_in_jupyter=True, auto_open=False, height=560)

## 6. Gap 策略回测

跳空可以做延续，也可以做回补。当前样本里延续版本更好：`gap_up` 后做多，`gap_down` 后做空；反向跳空会触发原方向离场，同时设置 3% 止盈、1.5% 止损，持仓用 `fp.vol_pms` 归一化。

In [5]:
TAKE_PROFIT = 0.03
STOP_LOSS = 0.015


def make_two_sided_strategy(indicator_cols, open_long_raw, open_short_raw):
    """用当前指标生成完整多空策略；持仓用 fp.vol_pms 做品种/波动率尺度归一化。"""
    return (
        col
        .with_cols(indicator_cols)
        .with_cols(
            open_long_raw.fill_null(col.lit(False)).alias("open_long_raw"),
            open_short_raw.fill_null(col.lit(False)).alias("open_short_raw"),
        )
        # 指标在当前 K 线收盘后才确认，所以入场信号后移一根 K 线，避免同根 K 线偷看。
        .with_cols(
            col("open_long_raw").shift(1).expanding().fill_null(col.lit(False)).alias("open_long_sig"),
            col("open_short_raw").shift(1).expanding().fill_null(col.lit(False)).alias("open_short_sig"),
        )
        .with_cols(
            col("open_long_sig", "close").stra.exit_by_pct(TAKE_PROFIT, False).expanding().alias("take_profit_long"),
            col("open_long_sig", "close").stra.exit_by_pct(STOP_LOSS, True).expanding().alias("stop_loss_long"),
            col("open_short_sig", "close").stra.exit_by_pct(TAKE_PROFIT, True).expanding().alias("take_profit_short"),
            col("open_short_sig", "close").stra.exit_by_pct(STOP_LOSS, False).expanding().alias("stop_loss_short"),
        )
        .with_cols(
            (col("take_profit_long") | col("stop_loss_long") | col("open_short_sig"))
                .fill_null(col.lit(False))
                .alias("exit_long_sig"),
            (col("take_profit_short") | col("stop_loss_short") | col("open_long_sig"))
                .fill_null(col.lit(False))
                .alias("exit_short_sig"),
        )
        .with_cols(
            col("open_long_sig", "exit_long_sig", "open_short_sig", "exit_short_sig")
                .stra.to_hold_two_sides()
                .expanding()
                .alias("hold")
        )
        .with_cols(
            (col("hold") / col.all.fp.vol_pms()).alias("hold")
        )
        .with_cols(col("close", "hold").bt.price(fee_rate=0.0).expanding())
        .over("ticker", "ct")
        .select(
            col("pnl")
                .sum()
                .group_by(col("datetime").dt.date().alias("date"))
                .batch.sort("date")
                .with_cols(col("pnl").sum().expanding().alias("pnl_cum"))
                .select("date", "pnl", "pnl_cum")
        )
    )


def calc_strategy_stats(strategy_daily: pl.DataFrame) -> pl.DataFrame:
    return col(
        col("date").first_value().alias("start_date"),
        col("date").last_value().alias("end_date"),
        col.lit(1).sum().alias("days"),
        col("pnl").sum().alias("total_pnl"),
        col("pnl").mean().alias("mean_daily_pnl"),
        col("pnl").std().alias("std_daily_pnl"),
        (col("pnl").mean() / col("pnl").std() * col.lit(252 ** 0.5)).alias("sharpe_like"),
        col("pnl").min().alias("worst_day_pnl"),
        col("pnl").max().alias("best_day_pnl"),
    ).calc_data(strategy_daily)

indicator_cols = col("open", "high", "low", "close").investopedia.gap()
strategy_daily_expr = make_two_sided_strategy(
    indicator_cols,
    col("gap_up"),
    col("gap_down"),
)
strategy_daily = strategy_daily_expr.calc_data(raw)
strategy_stats = calc_strategy_stats(strategy_daily)

print("strategy_daily shape:", strategy_daily.shape)
strategy_stats


strategy_daily shape: (859, 3)


start_date,end_date,days,total_pnl,mean_daily_pnl,std_daily_pnl,sharpe_like,worst_day_pnl,best_day_pnl
date,date,i32,f64,f64,f64,f64,f64,f64
2022-01-04,2024-12-31,859,71.573424,0.083908,1.515687,0.878807,-6.204445,7.666146


In [6]:
strategy_daily.tail(12)


date,pnl,pnl_cum
date,f64,f64
2024-12-18,0.064648,65.430473
2024-12-19,-0.428085,65.002388
2024-12-20,0.085235,65.087622
2024-12-21,0.125345,65.212968
2024-12-23,2.214603,67.427571
2024-12-24,2.504005,69.931576
2024-12-25,1.541438,71.473014
2024-12-26,0.827008,72.300022
2024-12-27,0.687042,72.987064


## 7. 策略 PnL 曲线

下面用 qust monitor 同时画累计 PnL 和每日 PnL。累计曲线显示这套规则跨合约、跨日期后的整体资金变化；每日柱状图用来观察收益是否集中在少数日期。

In [7]:
pnl_dashboard = col(
    col("date", "pnl_cum")
        .monitor("strategy_pnl_cum", show_axis_label=True)
        .line(),
    col("date", "pnl")
        .monitor("strategy_daily_pnl", show_axis_label=True)
        .bar(),
).monitor.make_monitor("black").monitor.add_grid([
    ["strategy_pnl_cum"],
    ["strategy_daily_pnl"],
]).runtime()

pnl_dashboard.plot(strategy_daily, open_in_jupyter=True, auto_open=False, height=640)


## 8. 使用时的注意事项

- 技术指标只能把价格结构转成可计算规则，不等于确定性交易建议。
- 形态类指标通常需要后续 K 线确认；如果用于实时交易，应把确认延迟纳入回测。
- 参数越敏感，信号越多但噪声越大；参数越保守，信号更少但滞后更明显。
- 在多合约或多股票数据上使用时，优先写 `.over("ticker", "ct")` 或合适的分组键。